In [1]:
# LOADING MODULES
import AtomicSnap
from AtomicSnap import AtomicSnapshots
import numpy as np
import os
import ase, ase.io, ase.visualize, ase.geometry.analysis
import matplotlib
import matplotlib.pyplot as plt
import os, sys
import scipy, scipy.stats

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
import json
import MDAnalysis
import MDAnalysis.analysis
import MDAnalysis.analysis.rdf
import MDAnalysis.analysis.msd

Conductivity module | Sorry no Julia found. In case you want to use it try with python-jl
Vibrational module| Sorry no Julia found. In case you want to use it try with python-jl
It seems that Julia is not available. Try to run with python-jl

Test if Julia works...

No Julia found!



In [81]:
# Initialisation des inputs depuis des .traj (list of ase objects)
atoms1 = ase.io.read("/home/azavadil/PhD/simulations/50EC/NVT/mace-omat-0-medium/traj.traj",index=":")
# atoms2 = ase.io.read("/home/azavadil/PhD/simulations/1Zn_2Cl_280H2O/NVT/mace-omat-0-medium/run01bis/traj.traj", index=":")
# atoms3 =ase.io.read("/home/azavadil/PhD/simulations/1Mg_2Cl_280H2O/NVT/mace-omat-0-medium/run01bis/traj.traj",index=":")
liste = [atoms1]


In [82]:
# Creation des universes pour analyses (Universe Object de MDAnalysis)
liste_universe = []
for i in range(len(liste)):
    # Retrieve topology informations
    n_atoms = len(liste[i][0])
    n_frames = len(liste[i])
    symbols = liste[i][0].get_chemical_symbols()
    masses = liste[i][0].get_masses()
    
    # Create empty universe
    u = MDAnalysis.Universe.empty(n_atoms,trajectory=True)
    
    # Add topology
    u.add_TopologyAttr("names", symbols)
    u.add_TopologyAttr("types", symbols)
    u.add_TopologyAttr("masses", masses)
    
    # Add trajectory
    coords = np.array([traj.positions for traj in liste[i]])
    cell = list(liste[i][0].cell.lengths()) + list(liste[i][0].cell.angles())
    u.load_new(coords, order="fac")
    for ts in u.trajectory:
        ts.dimensions=cell
        ts.dt = 0.5*1e-3*200
    liste_universe.append(u)


In [21]:
# INTER_RDF_S for specific site and CN computation
# Run rdf calculation (retrieve a list of distances between xi and yi)
urdf =  MDAnalysis.analysis.rdf.InterRDF_s(liste_universe[0], [[liste_universe[0].select_atoms("name C"), liste_universe[0].select_atoms("name O")]], nbins=500, range=(0.1,10.0))
urdf.run(verbose = True)
usrdf = np.sum(urdf.results.rdf[0][0], axis=0)/len(liste_universe[0].select_atoms("name O"))
ui_peak=np.argmax(usrdf)
ui_min = ui_peak + np.argmin(usrdf[ui_peak:])
ucdf = urdf.get_cdf()
ucn = np.sum(ucdf[0][0],axis=0)

# rdf =  MDAnalysis.analysis.rdf.InterRDF_s(liste_universe[0], [[liste_universe[0].select_atoms("name Zn"), liste_universe[0].select_atoms("name Cl")]], nbins=500, range=(0.1,10.0))
# rdf.run(verbose = True)
# srdf = np.sum(rdf.results.rdf[0][0], axis=0)/len(liste_universe[0].select_atoms("name Cl"))
# i_peak=np.argmax(srdf)
# i_min = i_peak + np.argmin(srdf[i_peak:])
# cdf = rdf.get_cdf()
# cn = np.sum(cdf[0][0],axis=0)

/home/azavadil/cp2k_venv/venv/lib/python3.12/site-packages/MDAnalysis/analysis/rdf.py:682: DeprecationWarning: The `u` attribute is superflous and will be removed in MDAnalysis 3.0.0.
  warnings.warn(


  0%|          | 0/4855 [00:00<?, ?it/s]

In [66]:
# HOW TO FIND BACKINTER_RDF when multiple A 
liste = []
for i, gr in enumerate(urdf.results.rdf[0]):
    liste.append(np.average(gr,axis=0))
gr_final = np.average(liste,axis=0)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199


In [76]:
fig = plt.figure(figsize=(8, 8))
gs = gridspec.GridSpec(1, 2, figure = fig)
ax1 = fig.add_subplot(gs[0,0])
ax1.plot(urdf.results.bins, gr_final, color='darkviolet', label=f'Pic at {urdf.results.bins[ui_peak]:.2f}')
ax1.plot(urdf.results.bins,ucn, color ='darkviolet', linestyle='--', label=f'CN of {ucn[ui_min]:.2f}')
plt.xlabel(f"Zn-O distance (Å)",fontsize = 18)
plt.ylabel(f" RDF Zn-O", fontsize = 18)
plt.legend(fontsize=20)
# ax2 = fig.add_subplot(gs[0,1])
# ax2.plot(rdf.results.bins, srdf, color='green', label=f'Pic at {rdf.results.bins[i_peak]:.2f} ')
# ax2.plot(rdf.results.bins,cn, color ='green', linestyle='--', label=f'CN of {cn[i_min]:.2f}')
plt.legend(fontsize=20)
# plt.axhline(y=5, color="yellow")
plt.xlabel(f"Zn-Cl distance (Å)",fontsize = 18)
plt.ylabel(f" RDF Zn-Cl", fontsize = 18)
plt.tick_params(axis = 'both', labelsize = 18)
plt.show()

In [83]:
# INTER_RDF ! For solvent
# Run rdf calculation (retrieve a list of distances between xi and yi)

rdf =  MDAnalysis.analysis.rdf.InterRDF(liste_universe[0].select_atoms("name C"), liste_universe[0].select_atoms("name O"), norm="rdf", nbins=500, range=(0.1,10.0))
rdf.run(verbose = True)
# urdf =  MDAnalysis.analysis.rdf.InterRDF(liste_universe[1].select_atoms("name C"), liste_universe[1].select_atoms("name O"), nbins=500, range=(0.1,10.0))
# urdf.run(verbose = True)


i_peak=np.argmax(rdf.results.rdf)
# ui_peak=np.argmax(urdf)


  0%|          | 0/5001 [00:00<?, ?it/s]

In [85]:
plt.plot(rdf.results.bins, rdf.results.rdf, color='darkviolet', label = f"RDF C-O of pure G1 with mace-omat\nPic at {rdf.results.bins[i_peak]:.2f}")
# plt.plot(urdf.results.bins, urdf.results.rdf, color='darkviolet', label=r'RDF C-O of G1 and Ca$\mathrm{Cl_{2}}$ with AIMD')
plt.legend(fontsize=20)
# plt.axhline(y=5, color="yellow")
plt.xlabel(f"C-O distance (Å)",fontsize = 18)
plt.ylabel(f" RDF C-O", fontsize = 18)
plt.tick_params(axis = 'both', labelsize = 18)
plt.show()

In [78]:
plt.plot(urdf.results.bins, usrdf, color='darkviolet', label=f'dt=0.5 RDF {atoms1[0].get_chemical_symbols()[0]}-O pic at {urdf.results.bins[ui_peak]:.2f}')
plt.plot(urdf.results.bins,ucn, color ='darkviolet', linestyle='--', label=f'CN of {ucn[ui_min]:.2f}')
#plt.plot(rdf.results.bins, srdf, color='green', label=f'dt= 0.25 fs')
#plt.plot(rdf.results.bins,cn, color ='green', linestyle='--', label=f'CN')
plt.legend(fontsize=20)
plt.tick_params(axis = 'both', labelsize = 18)
plt.show()

In [35]:
MSD = MDAnalysis.analysis.msd.EinsteinMSD(liste_universe[2], select='name H O', msd_type='xyz', fft=True)
MSD.run()
msd =  MSD.results.timeseries
lagtimes = MSD.results.delta_t_values

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 840/840 [00:00<00:00, 1394.48it/s]


In [36]:
nframes = MSD.n_frames
fig = plt.figure()
ax = plt.axes()
# plot the actual MSD
ax.plot(lagtimes, msd, color="black", ls="-", label=r'3D random walk')
exact = lagtimes*6
# plot the exact result
# ax.plot(lagtimes, exact, color="black", ls="--", label=r'$y=2 D\tau$')
plt.show()

In [37]:
from scipy.stats import linregress
start_time,  end_time  = lagtimes[len(lagtimes)//2] - 0.3 * lagtimes[-1], lagtimes[len(lagtimes)//2] + 0.3 * lagtimes[-1]

mask = (start_time < lagtimes) & (lagtimes < end_time)
linear_model = linregress(lagtimes[mask], msd[mask])
slope = linear_model.slope
error = linear_model.stderr
# dim_fac is 3 as we computed a 3D msd with 'xyz'
D = slope * 1/(2*MSD.dim_fac)
print(D)

0.004388026241409516


In [38]:
exa=lagtimes
plt.loglog(lagtimes, msd, label = f"D = {D} {r"$A^{2}\,\mathrm{ps}^{-1}$"}")
plt.loglog(lagtimes, exa)
plt.title("msd H2O")
plt.legend(fontsize=20)
plt.show()

In [7]:
print(liste_universe[0].trajectory.dt)

0.1


In [3]:
ana_tool = ase.geometry.analysis.Analysis(atoms1)
ana_tool2 = ase.geometry.analysis.Analysis(atoms2)

In [4]:
OCCAngles = ana_tool.get_angles("O","C","C", unique=True)
OCCAngles2 = ana_tool2.get_angles("O","C","C", unique=True)

In [9]:
len(OCCAngles2[0])

100

In [5]:
OCCAnglevalues = ana_tool.get_values(OCCAngles)
OCCAnglevalues2 = ana_tool2.get_values(OCCAngles2)


In [6]:
OCCAnglevalues_flatten = np.ravel(OCCAnglevalues)
OCCAnglevalues2_flatten = np.ravel(OCCAnglevalues2)

In [46]:
plt.hist(OCCAnglevalues_flatten, bins=100, label="mace-omat-0-m")
plt.hist(OCCAnglevalues2_flatten, bins=100,color="red", label="PBE")
plt.title("distribution of the OCC angle of glyme 1",fontsize=20)
plt.xlabel(r"$\theta$")
plt.ylabel("count")
plt.axvline(mean_omat,linestyle='--',color='darkblue')
plt.axvline(mean_PBE,linestyle='--',color="darkred")
plt.text(mean_omat-2.5,plt.ylim()[1]*0.9, rf"$\mu_{{omat}} = {mean_omat:.2f}°$",color="darkblue", ha='center',fontsize=12)
plt.text(mean_omat+2,plt.ylim()[1]*0.9, rf"$\mu_{{PBE}} = {mean_PBE:.2f}°$",color="darkred", ha='center',fontsize=12)
plt.legend(fontsize=16)
plt.show()

In [8]:
mean_omat= np.average(OCCAnglevalues)
mean_PBE = np.average(OCCAnglevalues2)